# GARMIN DATASET: Convert json -> csv
Conversion from "json" to csv

In [ ]:
import pandas as pd
import os, json
from datetime import timedelta
from pathlib import Path

INDIR = "DATASET/2025_TourE"
OUTDIR = "outputs/2025_TourE"

In [ ]:
def parse(fullName, outdir=OUTDIR):
    
    fName = os.path.split( fullName )[-1]
    if fName.endswith( ".json" ):
        fName = fName[:-5]
    
    outdir = Path( outdir )
    if not outdir.exists():
        outdir.mkdir( parents=True )
    
    try:
        df = pd.read_json( fullName )
        record_df = df.loc[(df["frame_type"] == "data_message") & (df.name == "record")]
        values_df = pd.concat(
            [
                pd.DataFrame.from_records(l)[["name", "value"]]
                .set_index("name")
                .transpose()
                for l in record_df.fields
            ]
        )
        if values_df.shape[0] != 0:
            values_df.to_csv(os.path.join( outdir, f"{fName}.csv" ) )
            
            stats = {}

            values_df["timestamp"] = pd.to_datetime(values_df["timestamp"])
            stats["data_filename"] = fullName
            stats["t_start"] = values_df.timestamp.min()
            stats["t_stop"] = values_df.timestamp.max()
            stats["duration_sec"] = stats["t_stop"] - stats["t_start"]

            stats["t_start"] = stats["t_start"].isoformat()
            stats["t_stop"] = stats["t_stop"].isoformat()
            stats["duration_sec"] = stats["duration_sec"].total_seconds()

            if stats["duration_sec"] > 1800:
                stats.update(
                    pd.DataFrame.from_records(
                        df.loc[
                            (df.frame_type == "data_message") & (df.name == "file_id")
                        ].fields.values[0]
                    )[["name", "value"]]
                    .set_index("name")
                    .to_dict()["value"]
                )
            with open( os.path.join( outdir, f"{fName}-stat.json" ), "w" ) as f:
                json.dump(stats, f, indent=2)
    except ValueError:
        print(f"No points in {fullName}")
    except KeyError:
        print(f"Empty file {fullName}")

In [ ]:
for f in os.walk(INDIR):
    dirName = f[0]
    if os.path.isdir( dirName ):
        for fullName in f[2]:
            if fullName.endswith(".json"):
                outdir = os.path.join( OUTDIR, os.path.relpath( dirName, INDIR ) )
                print( f"PARSING {dirName}/{fullName} -> {outdir}" )
                parse( f"{dirName}/{fullName}", outdir )
    

## 2. Parse and Load all csv

In [ ]:
import pandas as pd
import os, json
from datetime import timedelta


MIN_DURATION_S = 1800
INDIR = "GarminRawData"

In [ ]:
def load_csv( fName, device ):
    df = pd.read_csv( fName )
    df['timestamp'] = pd.to_datetime(df.timestamp)
    if (df.timestamp.max() - df.timestamp.min()).total_seconds() > MIN_DURATION_S:
        df["device"] = device
        df['session'] = df.timestamp.min()
        df['day'] = df.timestamp.min().day
    else:
        df=pd.DataFrame()
    return df

In [ ]:
garmin_df = pd.DataFrame()

for f in os.walk(INDIR):
    dirName = f[0]
    if os.path.isdir( dirName ):
        for fullName in f[2]:
            if fullName.endswith(".csv"):
                print( f"LOADING {dirName}/{fullName}" )
                garmin_df = pd.concat( [garmin_df,  load_csv( f"{dirName}/{fullName}", os.path.split(dirName)[-1] )] )
    

## 3. EXPORT to "garmin_df.pk"

In [ ]:
import pickle

with open( "garmin_df.pk", "wb" ) as fw:
    pickle.dump( garmin_df, fw )